# Unsloth — fast, memory-lean LoRA/QLoRA finetuning

A refresher on **Unsloth**: a drop-in finetuning library that makes LoRA/QLoRA training **~2× faster and ~50–70% lighter on VRAM** with **zero loss in accuracy** — by hand-writing fused Triton kernels and manual backward passes for the hot paths (attention RoPE, RMSNorm, SwiGLU MLP, cross-entropy) instead of leaning on the generic autograd graph. Same recipe as Hugging Face QLoRA, same numbers out, fewer bytes and seconds in.

**Domain:** LLM Inference, Training & Optimization  ·  **recommended addition**  ·  **runnable:** yes  ·  _cross-ref [QLoRA](./qlora.ipynb), [LoRA / ControlNet](./lora-controlnet.ipynb), [TRL RLHF/DPO](./trl-rlhf-dpo.ipynb), [Flash Attention](./flash-attention.ipynb)_

## 1. What & Why

You already know **QLoRA** (4-bit frozen base + small trainable LoRA adapters) is how you finetune a big model on one GPU. Unsloth answers the next question: *given that recipe, how do you make it run twice as fast on the same card without changing the math?*

The stock Hugging Face stack (`transformers` + `peft` + `bitsandbytes`) is correct but leaves performance on the table: each layer's RoPE, RMSNorm, SwiGLU, attention, and the final cross-entropy are built from many small PyTorch ops, each reading and writing big tensors to GPU memory, and autograd stores every intermediate for the backward pass. On a memory-bandwidth-bound workload that's where the time and VRAM go.

Unsloth rewrites exactly those hot spots:

1. **Fused Triton kernels** for RoPE, RMSNorm, SwiGLU/MLP, and cross-entropy — one kernel does in a single memory pass what was a dozen ops, slashing reads/writes.
2. **Hand-written manual backward passes** that recompute cheap things instead of storing them, and avoid the redundant upcasts/temporaries the generic graph creates.
3. **Memory tricks**: chunked cross-entropy (never materialize the full `tokens × vocab` logits in fp32), Unsloth-flavored gradient checkpointing that offloads activations to CPU, and keeping the LoRA path factored.

Crucially, all of this is **exact** — the outputs are bit-for-bit the same training math, just computed more efficiently. That's the headline: speed/memory wins with **no accuracy trade-off** (unlike, say, a lower rank or more aggressive quantization, which *do* trade quality).

**Reach for it when:** you're doing LoRA/QLoRA SFT/DPO on a **single** GPU (a Colab T4, a 3090/4090, one A100) and want it faster or to fit a bigger model/seq-len. **Skip it when:** you need multi-GPU/multi-node distributed training (Unsloth's free tier is single-GPU), an architecture it hasn't kernel-optimized, or you're doing full fine-tuning rather than adapter training.

## 2. Mental Model

Think of standard QLoRA training as **cooking a recipe by dirtying a fresh pan for every single step** — chop in one bowl, whisk in another, transfer, reheat. Each transfer (tensor read/write to VRAM) is slow, and you keep every dirty dish around in case you need to retrace your steps (autograd saving intermediates). Unsloth is the line cook who **fuses steps into one pan in one motion** and only keeps the dishes actually needed for cleanup — same dish on the plate, far less time and counter space.

```
Standard HF path (per transformer layer, simplified):
  x ─► [RMSNorm] ─► tmp ─► [RoPE] ─► tmp ─► [attn] ─► tmp ─► [RMSNorm] ─► tmp ─► [SwiGLU] ─► out
        ↑ each [op] = its own kernel launch + full read/write of a big tensor to HBM
        ↑ autograd saves every tmp for backward  ⇒  bandwidth- & memory-bound

Unsloth path:
  x ─► [ fused RMSNorm+RoPE kernel ] ─► [ fused SwiGLU kernel ] ─► out
        ↑ one kernel, one pass over memory, custom backward recomputes the cheap bits
        ↑ same numbers out, ~2× faster, ~½ the VRAM
```

Two ideas make it click:

- **The bottleneck is memory bandwidth, not FLOPs.** LLM training spends much of its time moving tensors to and from GPU HBM. Fusing N ops into 1 kernel turns N round-trips into 1 — that's the speedup, and it's free accuracy-wise.
- **You don't have to store what you can cheaply recompute.** A custom backward pass can regenerate an intermediate (e.g. re-apply RoPE) more cheaply than the bandwidth cost of having stored it — trading a little compute for a lot of saved memory.

One sentence: **Unsloth keeps QLoRA's math identical but replaces the generic op-by-op execution with fused, hand-differentiated kernels, so the same training run is bandwidth-efficient instead of bandwidth-bound.**

## 3. Key Concepts

| Term | What it means |
|------|---------------|
| **Fused kernel** | One GPU kernel that performs several logical ops (e.g. RMSNorm → scale → RoPE) in a single pass over memory, instead of launching one kernel per op. Fewer HBM round-trips = faster. |
| **Triton** | OpenAI's Python-like DSL for writing GPU kernels. Unsloth's custom kernels are Triton, so they're portable across NVIDIA cards without hand-written CUDA. |
| **Manual backward** | Unsloth hand-derives the gradient for its fused ops instead of relying on autograd's op-by-op graph, letting it skip redundant intermediates and upcasts. |
| **Chunked cross-entropy** | Compute the LM-head loss over the vocab in row-chunks so the giant `tokens × vocab` logits tensor is never fully materialized in fp32 — a major VRAM saver (demoed below). |
| **`FastLanguageModel`** | Unsloth's entry point: `from_pretrained(...)` loads a (often pre-quantized 4-bit) model with the kernels patched in; `get_peft_model(...)` attaches LoRA adapters. |
| **Pre-quantized `*-bnb-4bit` models** | Unsloth hosts ready 4-bit checkpoints (e.g. `unsloth/llama-3.2-1b-bnb-4bit`) so you skip the download-then-quantize step. |
| **`use_gradient_checkpointing="unsloth"`** | Unsloth's offloaded gradient checkpointing — streams activations to CPU RAM to fit longer sequences / bigger batches, beyond vanilla checkpointing. |
| **Exact, not approximate** | Unsloth's optimizations don't change the result (no extra quality loss). Speed/VRAM come from execution, not from lowering precision or rank. |
| **Single-GPU focus** | The open-source library targets one GPU. Multi-GPU/distributed is on the roadmap / Pro tier, not the free path. |

Mostly this builds directly on the **QLoRA** notebook: rank `r`, `alpha`, `target_modules`, NF4 4-bit base, paged optimizers — all the same knobs. Unsloth changes *how* that recipe executes, not *what* it is.

## 4. Setup

Unsloth's real kernels are **Triton on an NVIDIA GPU** — there is no production CPU path, and it pins specific `torch`/CUDA builds. So the executable examples below are **pure NumPy (CPU)** that reproduce the two ideas where the understanding lives — keeping the LoRA path factored, and chunked cross-entropy — and the real `FastLanguageModel` recipe (§5, Example 3) is **gated** behind an `os.getenv` + GPU check so the notebook still runs top-to-bottom anywhere.

```bash
# On a CUDA GPU (Colab/Linux). Unsloth pins torch/CUDA; follow its installer:
pip install unsloth
# trl + transformers + peft + bitsandbytes come in as dependencies for the QLoRA recipe.
```

The probe cell below just reports what's present in *this* kernel; the NumPy examples run regardless.

In [1]:
import importlib.util
import sys

import numpy as np


def have(mod: str) -> str:
    return "installed" if importlib.util.find_spec(mod) else "not installed"


print(f"python       : {sys.version.split()[0]}")
print(f"numpy        : {np.__version__}")
for m in ("torch", "unsloth", "transformers", "peft", "bitsandbytes", "trl"):
    print(f"{m:<13}: {have(m)}")

print("\nExamples 1 & 2 below are pure NumPy (CPU) and run regardless of the above.")

python       : 3.13.7
numpy        : 2.5.0
torch        : installed
unsloth      : not installed
transformers : installed
peft         : not installed
bitsandbytes : not installed
trl          : not installed

Examples 1 & 2 below are pure NumPy (CPU) and run regardless of the above.


## 5. Worked Examples

### Example 1 — keep the LoRA path *factored* (don't materialize ΔW)

A LoRA layer adds `ΔW = B·A` (rank `r`) to a frozen `W`. The naive way is to build the full `d×d` matrix `ΔW = B·A` and multiply by it; the efficient way — what Unsloth's fused LoRA path does — keeps the rank-`r` bottleneck and never materializes the big matrix. Same output (it's just associativity of matmul: `x·(B·A)ᵀ = (x·Aᵀ)·Bᵀ`), but vastly fewer FLOPs and no `d×d` temporary. This is the simplest instance of *same math, cheaper execution*.

In [2]:
import time

import numpy as np

rng = np.random.default_rng(0)
d, r, T = 4096, 16, 512  # hidden dim, LoRA rank, number of tokens
x = rng.standard_normal((T, d)).astype(np.float32)
A = (rng.standard_normal((r, d)) * 0.01).astype(np.float32)  # down-project (r x d)
B = (rng.standard_normal((d, r)) * 0.01).astype(np.float32)  # up-project   (d x r)

# Naive: materialize the full delta-W (d x d), then one big matmul.
t0 = time.perf_counter()
dW = B @ A               # (d, d) -- a 4096x4096 intermediate per layer!
y_materialized = x @ dW.T
t_mat = time.perf_counter() - t0

# Factored: never build dW; ride the rank-r bottleneck.
t0 = time.perf_counter()
y_factored = (x @ A.T) @ B.T  # (T, r) then (T, d)
t_fac = time.perf_counter() - t0

print(f"max abs diff        : {np.abs(y_materialized - y_factored).max():.2e}  (numerically identical)")
print(f"materialized dW     : {t_mat * 1e3:6.1f} ms, builds a {dW.shape} intermediate")
print(f"factored (rank {r})  : {t_fac * 1e3:6.1f} ms")
print(f"big-matmul FLOP cut  : ~{d / (2 * r):.0f}x fewer multiply-adds kept factored")

max abs diff        : 4.32e-07  (numerically identical)
materialized dW     :   16.5 ms, builds a (4096, 4096) intermediate
factored (rank 16)  :    1.0 ms
big-matmul FLOP cut  : ~128x fewer multiply-adds kept factored


### Example 2 — chunked cross-entropy: identical loss, a fraction of the memory

The final LM-head loss multiplies hidden states `(T, h)` by the output embedding `(h, V)` to get a `(T, V)` logits tensor — for a 32k vocab that's the single biggest activation in the model, and autograd wants it in fp32. Unsloth's fused cross-entropy computes the loss in **row chunks**, so the full `T×V` buffer is never alive at once. Below we compute the exact same loss both ways and compare peak logits memory.

In [3]:
import numpy as np

rng = np.random.default_rng(1)
T, V, h = 1024, 32000, 256  # tokens, vocab size, hidden dim
H = rng.standard_normal((T, h)).astype(np.float32)        # hidden states
W = (rng.standard_normal((h, V)) * 0.02).astype(np.float32)  # LM head / output embedding
targets = rng.integers(0, V, size=T)


def ce_full(H, W, targets):
    logits = H @ W                                   # (T, V) -- the giant intermediate
    logits = logits - logits.max(1, keepdims=True)   # numerical stability
    logp = logits - np.log(np.exp(logits).sum(1, keepdims=True))
    loss = -logp[np.arange(len(H)), targets].mean()
    return loss, logits.nbytes


def ce_chunked(H, W, targets, chunk=128):
    total, peak = 0.0, 0
    for i in range(0, len(H), chunk):
        sl = slice(i, i + chunk)
        logits = H[sl] @ W                           # (chunk, V) only
        logits = logits - logits.max(1, keepdims=True)
        logp = logits - np.log(np.exp(logits).sum(1, keepdims=True))
        rows = np.arange(logits.shape[0])
        total += -logp[rows, targets[sl]].sum()
        peak = max(peak, logits.nbytes)
    return total / len(H), peak


loss_full, mem_full = ce_full(H, W, targets)
loss_chunk, mem_chunk = ce_chunked(H, W, targets)
print(f"full    loss = {loss_full:.5f}   peak logits = {mem_full / 1e6:6.1f} MB")
print(f"chunked loss = {loss_chunk:.5f}   peak logits = {mem_chunk / 1e6:6.1f} MB")
print(f"identical loss, {mem_full / mem_chunk:.0f}x smaller logits buffer (chunk=128)")

full    loss = 10.42663   peak logits =  131.1 MB
chunked loss = 10.42663   peak logits =   16.4 MB
identical loss, 8x smaller logits buffer (chunk=128)


### Example 3 — the real Unsloth recipe (gated)

In practice you don't hand-roll any of the above — Unsloth patches the kernels in under a familiar API. The cell shows the exact, current shape: load a (pre-quantized 4-bit) model with `FastLanguageModel`, attach LoRA with `get_peft_model`, then hand the model to `trl`'s `SFTTrainer` like any PEFT model. It's **gated behind `RUN_UNSLOTH`** + a CUDA + install check so the notebook still executes on a plain CPU.

In [4]:
import os

HAVE_UNSLOTH = importlib.util.find_spec("unsloth") is not None
_have_torch = importlib.util.find_spec("torch") is not None
GPU = _have_torch and __import__("torch").cuda.is_available()

if os.getenv("RUN_UNSLOTH") and HAVE_UNSLOTH and GPU:
    from unsloth import FastLanguageModel

    # 1) Load a pre-quantized 4-bit base with Unsloth's kernels patched in.
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/llama-3.2-1b-bnb-4bit",  # ready 4-bit checkpoint
        max_seq_length=2048,
        load_in_4bit=True,   # QLoRA-style frozen 4-bit base
        dtype=None,          # auto: bf16 on Ampere+, else fp16
    )

    # 2) Attach LoRA adapters (same knobs as the QLoRA notebook).
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        lora_dropout=0,           # 0 lets Unsloth take a faster fused path
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth",  # offloaded checkpointing for long seqs
    )
    model.print_trainable_parameters()
    # 3) Hand `model` + `tokenizer` to trl.SFTTrainer exactly as a normal PEFT model.
else:
    print("Skipping GPU path (set RUN_UNSLOTH=1 on a CUDA box with unsloth). Recipe shape:\n")
    print(
        "from unsloth import FastLanguageModel\n"
        "model, tok = FastLanguageModel.from_pretrained(\n"
        "    'unsloth/llama-3.2-1b-bnb-4bit', max_seq_length=2048, load_in_4bit=True)\n"
        "model = FastLanguageModel.get_peft_model(\n"
        "    model, r=16, lora_alpha=32, lora_dropout=0,\n"
        "    target_modules=['q_proj','k_proj','v_proj','o_proj',\n"
        "                    'gate_proj','up_proj','down_proj'],\n"
        "    use_gradient_checkpointing='unsloth')\n\n"
        "# then the usual trl loop -- nothing Unsloth-specific past this point:\n"
        "from trl import SFTTrainer, SFTConfig\n"
        "trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds,\n"
        "                     args=SFTConfig(per_device_train_batch_size=2,\n"
        "                                    optim='adamw_8bit', max_seq_length=2048))\n"
        "trainer.train()\n"
        "model.save_pretrained('my-lora')           # tiny adapter\n"
        "model.save_pretrained_gguf('out', tok)     # or export merged GGUF for llama.cpp"
    )

Skipping GPU path (set RUN_UNSLOTH=1 on a CUDA box with unsloth). Recipe shape:

from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained(
    'unsloth/llama-3.2-1b-bnb-4bit', max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth')

# then the usual trl loop -- nothing Unsloth-specific past this point:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds,
                     args=SFTConfig(per_device_train_batch_size=2,
                                    optim='adamw_8bit', max_seq_length=2048))
trainer.train()
model.save_pretrained('my-lora')           # tiny adapter
model.save_pretrained_gguf('out', tok)     # or export merged GGUF for llama.cpp


## 6. Gotchas & Pitfalls

- **It's single-GPU (free tier).** Unsloth shines on *one* card. If you need DDP/FSDP across multiple GPUs or nodes, the open-source path doesn't cover it — reach for `trl`+`accelerate`/`axolotl` with FSDP, or Unsloth Pro.
- **Pinned `torch`/CUDA versions.** The kernels are compiled against specific builds; Unsloth's installer picks the matching wheel. Installing it into an environment with a mismatched torch is the #1 source of import/CUDA errors. Use a fresh env or follow the official install matrix.
- **Architecture coverage is finite.** Unsloth hand-writes kernels per model family (Llama, Mistral, Gemma, Qwen, Phi, …). A brand-new architecture it hasn't patched yet falls back to slow paths or isn't supported — check the supported-models list before assuming the speedup.
- **`import unsloth` must come first.** It monkey-patches `transformers`/`peft` at import time. Import it *before* the libraries it patches, or the optimized paths won't be installed and you'll silently get stock speed.
- **The 2× / 70% numbers are for LoRA/QLoRA SFT on supported models**, measured against the stock HF stack with similar settings. Your mileage depends on seq-len, batch, GPU, and which kernels your model hits — treat them as ballpark, not guarantees.
- **`lora_dropout=0` is the fast path.** Nonzero dropout disables one of Unsloth's fused shortcuts. If you don't specifically need adapter dropout, leave it at 0 for the full speedup.
- **It optimizes adapter training, not full fine-tuning.** The wins are built around the LoRA/QLoRA recipe. Full-parameter fine-tuning is outside its sweet spot.
- **Exporting:** use `save_pretrained` for the tiny adapter, `save_pretrained_merged` to fold it into the base, or `save_pretrained_gguf` to emit a GGUF for llama.cpp/Ollama. Don't re-upload the multi-GB base — ship the adapter.
- **Same QLoRA gotchas still apply.** Rank/alpha scaling (`alpha/r`), `target_modules` as a quality lever, can't merge into a 4-bit base — see the QLoRA notebook; Unsloth doesn't change any of that.

## 7. When to Use vs Alternatives

| Option | Speed / VRAM vs stock | Multi-GPU | Accuracy | Best for |
|--------|----------------------|-----------|----------|----------|
| **Unsloth** | ~2× faster, ~50–70% less VRAM | Single-GPU (free) | **Exact** (no loss) | LoRA/QLoRA SFT/DPO on one GPU, fast iteration, Colab |
| **HF `transformers`+`peft`+`bitsandbytes`** | baseline | Yes (accelerate) | Exact | The reference QLoRA stack; max model/arch coverage |
| **Axolotl** | config-driven, FSDP/DeepSpeed | Yes | Exact | YAML-configured multi-GPU runs, many recipes |
| **TRL `SFTTrainer`/`DPOTrainer`** | baseline trainer | Yes | Exact | The training *loop* (Unsloth/peft plug into it) |
| **torchtune** | native PyTorch recipes | Yes | Exact | PyTorch-idiomatic, distributed finetuning |

**Rules of thumb:**
- One GPU, doing LoRA/QLoRA, want it faster or to fit a bigger model/seq-len → **Unsloth** (it's a near-free win since the math is unchanged).
- Need multi-GPU / multi-node distributed training → **Axolotl** or **TRL/torchtune + FSDP/DeepSpeed**; not free-tier Unsloth.
- Model architecture Unsloth hasn't kernel-optimized → fall back to the **stock HF stack** (still correct, just slower).
- You want the training *loop* (SFT/DPO/PPO) — that's **TRL**; Unsloth supplies the optimized *model*, TRL drives the steps. They compose.
- Prefer declarative YAML over Python → **Axolotl**; prefer a patched-model + your own loop → **Unsloth**.

Note Unsloth is **complementary** to TRL: a typical fast SFT is "Unsloth model + TRL `SFTTrainer`." It competes with the *plain* `transformers`/`peft` execution path, not with the trainer.

## 8. Resources

- **Unsloth GitHub** — source, supported models, benchmarks, install matrix: <https://github.com/unslothai/unsloth>
- **Unsloth docs** — finetuning guides, saving/GGUF export, config reference: <https://docs.unsloth.ai>
- **Official example notebooks** (Colab, per model family) — copy-paste starting points: <https://github.com/unslothai/unsloth#-finetune-for-free>
- **Triton** — the GPU-kernel DSL Unsloth's fused kernels are written in: <https://triton-lang.org>
- **QLoRA paper** — Dettmers et al., the 4-bit + LoRA recipe Unsloth accelerates: <https://arxiv.org/abs/2305.14314>
- **TRL `SFTTrainer`** — the trainer Unsloth models plug into: <https://huggingface.co/docs/trl/sft_trainer>

**Cross-refs in this library:** [QLoRA](./qlora.ipynb) (the recipe Unsloth speeds up), [LoRA / ControlNet](./lora-controlnet.ipynb) (the low-rank adapter idea), [TRL RLHF/DPO](./trl-rlhf-dpo.ipynb) (the training loop you pair it with), [Flash Attention](./flash-attention.ipynb) (the same fuse-the-kernels, save-bandwidth idea applied to attention), [GGUF & llama.cpp](./gguf-llama-cpp.ipynb) (export a finished model for CPU inference).